# Train / val loss curves — STATE, Geneformer, SCVI

Sibling of `2026-04-17_11-32_plotting_state_loss_curves.ipynb` but covering all three learned algorithms across every dataset, size and quality.

Per-algo storage:
- **STATE** — `results/State/model/loss/metrics.csv` → columns `step`, `trainer/train_loss`, `validation/val_loss`.
- **Geneformer** — highest-numbered `results/Geneformer/checkpoint-NNNN/trainer_state.json` holds the full HuggingFace `log_history` (train `loss` + `eval_loss` per step) and `best_model_checkpoint`.
- **SCVI** — no per-step loss history on disk; the SCVI algo logs to Weights & Biases only, and locally keeps just `test_loss.txt` (final ELBO). SCVI is therefore reported in the completeness table but skipped in the per-step plots.

Marker semantics: `X` = checkpoint that would be (or was) loaded — for STATE it is the `argmin(val_loss)` row of the metrics CSV; for Geneformer it is the step reported in `best_model_checkpoint` (or `argmin(eval_loss)` if that field is empty).

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

DATA_ROOT = Path('/home/igor/noise_scaling/data')
DATASETS = ['PBMC', 'larry', 'merfish', 'shendure']
ALGOS = ['State', 'Geneformer', 'SCVI']

STATE_ES_PATIENCE = 5   # scaling_laws.algo.state.State — 1000-step val window
GF_ES_PATIENCE    = 3   # scaling_laws.prepare.data._get_config_for_size Geneformer

In [2]:
# ── Walk the data tree and load train/val curves per (algo, dataset, size, quality) ──
# curves[algo][dataset][size][quality] = {'train': DataFrame[step, loss],
#                                         'val':   DataFrame[step, loss],
#                                         'best_step': int, 'best_val': float,
#                                         'total_steps': int, 'epoch_max': int}
# SCVI stays empty (no curves on disk) but its presence/test_loss is still
# reported in the summary table below.

def _is_size_dir(p: Path) -> bool:
    return p.is_dir() and p.name.isdigit()

def _is_quality_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    try:
        float(p.name); return True
    except ValueError:
        return False

def _read_scalar(p: Path) -> float | None:
    try:
        return float(p.read_text().strip()) if p.exists() else None
    except Exception:
        return None

# ── STATE loader: Lightning metrics.csv ────────────────────────────────

def _load_state(model_dir: Path) -> dict | None:
    csv = model_dir / 'loss' / 'metrics.csv'
    if not csv.exists():
        # Lightning sometimes only keeps the live file under checkpoints/.
        ckpt_dir = model_dir / 'checkpoints'
        if ckpt_dir.is_dir():
            matches = list(ckpt_dir.glob('state_*/version_*/metrics.csv'))
            if matches:
                csv = matches[0]
    if not csv.exists():
        return None
    try:
        df = pd.read_csv(csv, on_bad_lines='skip')
    except Exception:
        return None
    needed = {'step', 'trainer/train_loss', 'validation/val_loss'}
    if not needed.issubset(df.columns):
        return None
    train = df[['step', 'trainer/train_loss']].dropna().rename(
        columns={'trainer/train_loss': 'loss'}).reset_index(drop=True)
    val = df[['step', 'validation/val_loss']].dropna().rename(
        columns={'validation/val_loss': 'loss'}).reset_index(drop=True)
    if train.empty or val.empty:
        return None
    bidx = val['loss'].idxmin()
    return {
        'train': train, 'val': val,
        'best_step': int(val.iloc[bidx]['step']),
        'best_val':  float(val.iloc[bidx]['loss']),
        'total_steps': int(df['step'].max()) if 'step' in df.columns else 0,
        'epoch_max':   int(df['epoch'].max()) if 'epoch' in df.columns and df['epoch'].notna().any() else -1,
        'csv':         csv,
    }

# ── Geneformer loader: HF trainer_state.json inside highest checkpoint ─

_CKPT_RE = re.compile(r'checkpoint-(\d+)$')

def _latest_geneformer_trainer_state(gf_root: Path) -> Path | None:
    """Return the trainer_state.json from the highest-numbered checkpoint dir.
    HF saves a *cumulative* log_history at every checkpoint, so the last one
    has the most data."""
    if not gf_root.is_dir():
        return None
    best = None
    best_step = -1
    for p in gf_root.iterdir():
        m = _CKPT_RE.search(p.name)
        if m and p.is_dir():
            step = int(m.group(1))
            ts = p / 'trainer_state.json'
            if ts.exists() and step > best_step:
                best_step, best = step, ts
    return best

def _load_geneformer(algo_root: Path) -> dict | None:
    ts = _latest_geneformer_trainer_state(algo_root)
    if ts is None:
        return None
    try:
        state = json.loads(ts.read_text())
    except Exception:
        return None
    log = state.get('log_history', []) or []
    if not log:
        return None
    train_rows = [(e['step'], e['loss']) for e in log if 'loss' in e and 'step' in e]
    val_rows   = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e and 'step' in e]
    if not train_rows or not val_rows:
        return None
    train = pd.DataFrame(train_rows, columns=['step', 'loss']).sort_values('step').reset_index(drop=True)
    val   = pd.DataFrame(val_rows,   columns=['step', 'loss']).sort_values('step').reset_index(drop=True)

    # best_model_checkpoint encodes the selected step via its suffix.
    best_ckpt = state.get('best_model_checkpoint') or ''
    m = _CKPT_RE.search(best_ckpt or '')
    if m:
        best_step = int(m.group(1))
        hit = val[val['step'] == best_step]
        best_val = float(hit.iloc[0]['loss']) if not hit.empty else float(val['loss'].min())
    else:
        bidx = val['loss'].idxmin()
        best_step = int(val.iloc[bidx]['step'])
        best_val  = float(val.iloc[bidx]['loss'])

    return {
        'train': train, 'val': val,
        'best_step': best_step, 'best_val': best_val,
        'total_steps': int(state.get('global_step') or train['step'].max()),
        'epoch_max':   int(state.get('epoch') or -1),
        'trainer_state': ts,
    }

# ── SCVI loader: no per-step history — only the final test ELBO ────────

def _load_scvi(model_dir: Path) -> dict | None:
    test_loss = _read_scalar(model_dir / 'test_loss.txt')
    if test_loss is None:
        return None
    return {
        'train': pd.DataFrame(columns=['step', 'loss']),
        'val':   pd.DataFrame(columns=['step', 'loss']),
        'best_step': None, 'best_val': test_loss,
        'total_steps': 0, 'epoch_max': -1,
        'no_curves': True,  # flag the plot can check
    }

LOADERS = {
    'State':      lambda qdir: _load_state((qdir / 'results' / 'State' / 'model')),
    'Geneformer': lambda qdir: _load_geneformer((qdir / 'results' / 'Geneformer')),
    'SCVI':       lambda qdir: _load_scvi((qdir / 'results' / 'SCVI' / 'model')),
}

curves: dict[str, dict[str, dict[int, dict[float, dict]]]] = {a: {} for a in ALGOS}
for ds in DATASETS:
    ds_dir = DATA_ROOT / ds
    if not ds_dir.is_dir():
        continue
    for size_dir in sorted(filter(_is_size_dir, ds_dir.iterdir()), key=lambda p: int(p.name)):
        size = int(size_dir.name)
        for q_dir in sorted(filter(_is_quality_dir, size_dir.iterdir()), key=lambda p: float(p.name)):
            q = float(q_dir.name)
            for algo, fn in LOADERS.items():
                info = fn(q_dir)
                if info is None:
                    continue
                curves[algo].setdefault(ds, {}).setdefault(size, {})[q] = info

# Summary counts.
for algo in ALGOS:
    n_runs = sum(len(qm) for ds in curves[algo] for qm in curves[algo][ds].values())
    n_sizes = sum(len(curves[algo][ds]) for ds in curves[algo])
    n_ds    = len(curves[algo])
    print(f'{algo:>11}: {n_runs:>5} runs across {n_sizes:>3} (dataset, size) combos in {n_ds} datasets')

      State:   400 runs across  40 (dataset, size) combos in 4 datasets
 Geneformer:   399 runs across  40 (dataset, size) combos in 4 datasets
       SCVI:   400 runs across  40 (dataset, size) combos in 4 datasets


In [3]:
# ── Completeness + sample-stats summary per algo ───────────────────────
# One row per (algo, dataset, size, quality) run that loaded successfully.
rows = []
for algo in ALGOS:
    for ds in curves[algo]:
        for size, qmap in curves[algo][ds].items():
            for q, info in qmap.items():
                rows.append({
                    'algo': algo, 'dataset': ds, 'size': size, 'quality': q,
                    'total_steps': info.get('total_steps', 0),
                    'epoch_max':   info.get('epoch_max', -1),
                    'n_train_pts': len(info.get('train', [])),
                    'n_val_pts':   len(info.get('val', [])),
                    'best_step':   info.get('best_step'),
                    'best_val':    info.get('best_val'),
                    'no_curves':   info.get('no_curves', False),
                })
summary_df = pd.DataFrame(rows).sort_values(['algo', 'dataset', 'size', 'quality']).reset_index(drop=True)
print(f'{len(summary_df)} (algo, dataset, size, quality) runs loaded')
display(summary_df.groupby(['algo', 'dataset'])['size'].nunique().unstack('dataset', fill_value=0).rename_axis('sizes per (algo, dataset)'))

1199 (algo, dataset, size, quality) runs loaded


dataset,PBMC,larry,merfish,shendure
"sizes per (algo, dataset)",,,,
Geneformer,10,10,10,10
SCVI,10,10,10,10
State,10,10,10,10


## Per-algo figure: rows = sizes, cols = datasets, stacked train + val

One figure per algorithm. Inside a figure: columns are datasets and rows alternate **train (top)** / **val (bottom)** for each size rank. Viridis encodes quality; `X` marker on the val row marks the checkpoint that would be loaded (`argmin(val)` for STATE, `best_model_checkpoint` for Geneformer).

SCVI has no per-step history on disk (logged only to wandb), so its figure prints a stub instead.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

PER_ROW_H = 2.2      # per-subplot vertical inches
FIG_DPI   = 300
TRAIN_SMOOTH_WIN = {'State': 200, 'Geneformer': 1}  # step-based rolling mean

best_marker = mlines.Line2D(
    [], [], color='k', marker='X', markersize=8,
    markerfacecolor='white', markeredgecolor='k',
    linestyle='None', label='selected checkpoint (best val)',
)


def plot_algo(algo: str):
    algo_curves = curves.get(algo, {})
    ds_list = [ds for ds in DATASETS if ds in algo_curves and algo_curves[ds]]
    if not ds_list:
        print(f'[{algo}] no curves available on disk — skipping (SCVI only keeps test_loss.txt; wandb is upstream)')
        return

    sizes_by_ds = {ds: sorted(algo_curves[ds].keys()) for ds in ds_list}
    max_ranks = max(len(sizes_by_ds[ds]) for ds in ds_list)
    ncols = len(ds_list)
    nrows = 2 * max_ranks  # train + val for each rank

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.2 * ncols, PER_ROW_H * nrows),
        dpi=FIG_DPI, squeeze=False,
    )
    cmap = plt.get_cmap('viridis')
    rank_norm = plt.Normalize(0, max(max_ranks - 1, 1))

    # Column headers
    for col, ds in enumerate(ds_list):
        axes[0, col].annotate(
            ds, xy=(0.5, 1.35), xycoords='axes fraction',
            ha='center', va='bottom', fontsize=14, fontweight='bold',
        )

    for rank in range(max_ranks):
        row_t = 2 * rank
        row_v = row_t + 1

        axes[row_t, 0].annotate(
            f'rank {rank + 1}/{max_ranks}',
            xy=(-0.32, -0.05), xycoords='axes fraction',
            ha='center', va='center', fontsize=11, fontweight='bold',
            color=cmap(rank_norm(rank)), rotation=90,
        )

        for col, ds in enumerate(ds_list):
            ax_t, ax_v = axes[row_t, col], axes[row_v, col]
            sizes = sizes_by_ds[ds]
            if rank >= len(sizes):
                for ax in (ax_t, ax_v):
                    ax.set_xticks([]); ax.set_yticks([])
                    for sp in ax.spines.values():
                        sp.set_visible(False)
                continue
            size = sizes[rank]
            qmap = algo_curves[ds][size]
            qs = sorted(qmap.keys())
            if not qs:
                for ax in (ax_t, ax_v):
                    ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                            transform=ax.transAxes, color='gray', fontsize=10)
                    ax.set_xticks([]); ax.set_yticks([])
                ax_t.set_title(f'N = {size:,}', fontsize=10)
                continue
            qnorm = (plt.Normalize(np.log10(min(qs)), np.log10(max(qs)))
                     if len(qs) > 1 else plt.Normalize(0, 1))
            smooth_win = TRAIN_SMOOTH_WIN.get(algo, 1)
            for q in qs:
                info = qmap[q]
                color = cmap(qnorm(np.log10(q))) if len(qs) > 1 else cmap(0.5)
                train = info['train'][info['train']['step'] > 0]
                val   = info['val'][info['val']['step'] > 0]
                if train.empty or val.empty:
                    continue
                if smooth_win > 1:
                    t_y = train['loss'].rolling(window=smooth_win, min_periods=10, center=True).mean()
                else:
                    t_y = train['loss']
                ax_t.plot(train['step'], t_y, color=color, linewidth=1.2, label=f'q={q:g}')
                ax_v.plot(val['step'], val['loss'],
                          color=color, marker='o', linewidth=1.4, markersize=4, label=f'q={q:g}')
                if info.get('best_step') is not None:
                    ax_v.scatter([info['best_step']], [info['best_val']],
                                 color=color, marker='X', s=55, edgecolor='k',
                                 linewidth=0.5, zorder=5)
            for ax in (ax_t, ax_v):
                ax.grid(alpha=0.3)
            ax_t.set_title(f'N = {size:,}', fontsize=10)
            ax_v.set_title('')
            if col == 0:
                ax_t.set_ylabel('train loss')
                ax_v.set_ylabel('val loss')
            if row_v == nrows - 1:
                ax_v.set_xlabel('optimizer step')
            if col == ncols - 1 and rank == max_ranks - 1:
                # Two legends on the final subplot: qualities + checkpoint marker.
                qleg = ax_v.legend(loc='upper right', fontsize=6, ncol=2, title='quality')
                ax_v.add_artist(qleg)
                ax_v.legend(handles=[best_marker], loc='lower right', fontsize=8, frameon=True)

    plt.suptitle(f'{algo} loss curves — all datasets × all sizes × all qualities',
                 y=1.003, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(left=0.07, top=0.97, hspace=0.5)
    plt.show()


for algo in ALGOS:
    plot_algo(algo)

### SCVI note

SCVI is trained with `WandbLogger` and no local metrics file is written — see `scaling_laws/src/scaling_laws/algo/scvi.py`. The summary table above still shows per-(size, quality) rows for SCVI using the final test ELBO (`test_loss.txt`), but there is nothing to plot as a step-by-step curve. If training curves are needed, pull them from the SCVI wandb project instead of disk.